# Weed Management Synthetic Evaluation Dataset

The process is:

1. Decide the dimensions of variation.
2. Ask an LLM to create scenario type tuples.
3. Ask an LLM to generate realistic farmer questions from those tuples.
4. Review and save the evaluation dataset.

The goal is to create diverse farmer-like questions for the weed management RAG chatbot.

In [1]:
from pathlib import Path
import json

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# Keep paths working whether this notebook is run from project root or notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
EVAL_DIR = DATA_DIR / "eval"
CHUNKS_PATH = DATA_DIR / "processed" / "chunks.json"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.4)

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
topics = [chunk["metadata"]["topic"] for chunk in chunks]

topics

['Weed',
 'This brochure was created to increase',
 'What are noxious weeds?',
 'Weed Control Methods',
 'Bull thistle',
 'Canada thistle',
 'Musk thistle',
 'Scotch thistle',
 'Diffuse knapweed',
 'Spotted knapweed',
 'Russian knapweed',
 'Houndstongue',
 'Common Mullein',
 'Locoweed (Wooly)',
 'Salt Cedar (Tamarisk)',
 'Hoary Cress (Whitetop)',
 'Kochia',
 'Russian Olive',
 'Perennial Pepperweed',
 'Field Bindweed',
 'Common Cocklebur',
 'Rabbit Brush',
 'Winter Annuals',
 'Backyard Weed Control Tips']

## Step 1: Decide the Dimensions of Variation and generate combinations

first define dimensions that should vary across inputs.

For this weed chatbot, the dimensions are:

- **Topic**: weed or general topic from the PDF.
- **Question type**: identification, lifecycle, herbicide timing, mechanical control, biological control, safety, or out-of-scope.
- **Farmer language style**: short, field-note style, confused, typo-heavy, or plain language.
- **Expected behavior**: answer from the PDF or say the PDF does not provide enough information.

These dimensions make the eval set more realistic than using only clean classroom-style questions.

## Generate dimension combinations
We can give the dimensions to an llm which will do permutation combination and create the tuples.


In [2]:
from langchain_core.prompts import PromptTemplate


SCENARIO_TYPE_PROMPT = """
I am designing a weed management question-answering chatbot and want to test it with a diverse set of realistic farmer scenarios.

Please generate 20 unique combinations (tuples) using the following key dimensions and their possible values:

- Topic: Musk thistle, Bull thistle, Canada thistle, Diffuse knapweed, Salt Cedar (Tamarisk), Field Bindweed, Weed Control Methods, Crabgrass / not in PDF
- Question Type: Weed identification, Lifecycle, Mechanical control, Herbicide timing, General weed control methods, Out-of-scope question
- Farmer Language Style: Concise practical question, Detailed field observation, Professional agronomy question, Follow-up after failed control
- Expected Behavior: Answer from PDF, Say PDF does not provide enough information

Each combination should select one value from each dimension.

Present the results as a list of tuples, where each tuple contains one value for each dimension in the following order:
(Topic, Question Type, Farmer Language Style, Expected Behavior)

Rules:
- Ensure the combinations are varied and realistic.
- Prioritize important chatbot behaviors: identification, lifecycle, control method, herbicide timing, and out-of-scope handling.
- Most combinations should be answerable from the PDF.
- Include at least 3 out-of-scope scenarios using "Crabgrass / not in PDF".
- Do not generate the actual farmer questions yet. Only generate the scenario tuples.
"""

weed_question_combinations = llm.invoke(SCENARIO_TYPE_PROMPT)

print(weed_question_combinations.content)

Here are 20 unique and varied scenario tuples for your weed management chatbot testing:

1. (Musk thistle, Weed identification, Concise practical question, Answer from PDF)  
2. (Bull thistle, Lifecycle, Detailed field observation, Answer from PDF)  
3. (Canada thistle, Mechanical control, Professional agronomy question, Answer from PDF)  
4. (Diffuse knapweed, Herbicide timing, Concise practical question, Answer from PDF)  
5. (Salt Cedar (Tamarisk), General weed control methods, Detailed field observation, Answer from PDF)  
6. (Field Bindweed, Weed identification, Professional agronomy question, Answer from PDF)  
7. (Musk thistle, Herbicide timing, Follow-up after failed control, Answer from PDF)  
8. (Bull thistle, Mechanical control, Concise practical question, Answer from PDF)  
9. (Canada thistle, General weed control methods, Detailed field observation, Answer from PDF)  
10. (Diffuse knapweed, Lifecycle, Professional agronomy question, Answer from PDF)  
11. (Salt Cedar (Tama

## Step 2: Generate Farmer Questions from Scenario Combinations

In the previous step, the LLM generated `weed_question_combinations`, where each combination describes a test scenario using dimensions such as topic, question type, farmer language style, and expected behavior.

Now we use those combinations as input to a second prompt that generates realistic farmer questions.

This is an example of **prompt chaining**:

1. **Prompt 1** generates structured scenario combinations.
2. **Prompt 2** uses those combinations to generate realistic user questions.

This helps create a more diverse evaluation dataset because the first prompt controls coverage across important dimensions, while the second prompt turns those dimensions into natural farmer-like questions.

In [3]:
SCENARIO_GENERATION_PROMPT = """
Convert these weed management scenario combinations into realistic farmer questions.

Each combination has this format:
(Topic, Question Type, Farmer Language Style, Expected Behavior)

Generate exactly 2 farmer questions for each combination.

Include variations in:
- Short direct questions
- Detailed field observations
- Practical treatment-planning questions
- Follow-up questions after failed control attempts
- Natural farmer language
- Occasional minor typos or informal wording

Rules:
- The questions should sound realistic for farmers, ranchers, landowners, or field users.
- Do not answer the questions.
- Do not include expected answers.
- For out-of-scope combinations, generate questions that the weed PDF likely cannot answer.
- Return the results in JSON format as a list of strings.
- Return only valid JSON.

<weed_question_combinations>
{weed_question_combinations}
</weed_question_combinations>
"""

prompt = SCENARIO_GENERATION_PROMPT.format(
    weed_question_combinations=weed_question_combinations.content
)

question_response = llm.invoke(prompt)

print(question_response.content)

[
  "What are the key features to spot musk thistle quickly in my pasture?",
  "How can I tell musk thistle apart from other thistles when scouting fields?",
  "I've noticed bull thistle plants with thick stems and purple flowers late in the season. Is this normal for their lifecycle?",
  "The bull thistle in my field seems to be flowering earlier than usual this year. Could this indicate a change in its growth stages?",
  "What mechanical methods work best to control Canada thistle without damaging my crops?",
  "Can tilling at certain times reduce Canada thistle spread effectively on my farm?",
  "When is the best time to spray herbicide on diffuse knapweed for maximum effect?",
  "Should I wait until diffuse knapweed starts flowering before applying herbicides, or is earlier better?",
  "The salt cedar along my riverbank is growing thick and spreading fast. What control methods could I use to manage it effectively?",
  "I've seen salt cedar resprouting after cutting. What integrated

In [5]:
import json

synthetic_questions = json.loads(question_response.content)

print(f"Generated {len(synthetic_questions)} questions")


Generated 25 questions


In [6]:
import json
from pathlib import Path


EVAL_DIR = Path("../data/eval") if Path.cwd().name == "notebooks" else Path("data/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS_JSON_PATH = EVAL_DIR / "synthetic_farmer_questions.json"
QUESTIONS_JSONL_PATH = EVAL_DIR / "synthetic_farmer_questions.jsonl"


# question_response.content should be valid JSON: a list of strings
synthetic_questions = json.loads(question_response.content)


QUESTIONS_JSON_PATH.write_text(
    json.dumps(synthetic_questions, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


with QUESTIONS_JSONL_PATH.open("w", encoding="utf-8") as file:
    for question in synthetic_questions:
        file.write(json.dumps({"question": question}, ensure_ascii=False) + "\n")


print(f"Saved {len(synthetic_questions)} questions")
print(f"JSON: {QUESTIONS_JSON_PATH}")
print(f"JSONL: {QUESTIONS_JSONL_PATH}")

Saved 25 questions
JSON: ../data/eval/synthetic_farmer_questions.json
JSONL: ../data/eval/synthetic_farmer_questions.jsonl
